In [5]:
# Cell 1: setup
import os, sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))

In [6]:
# Cell 2: confirm all 8 day files are in data/raw
from src.data_prep_m2 import check_files, run_m2_split
check_files()

Looking in /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project/data/raw
  [seen  ] Monday               Monday-WorkingHours.pcap_ISCX.csv
  [seen  ] Tuesday              Tuesday-WorkingHours.pcap_ISCX.csv
  [seen  ] Wednesday            Wednesday-workingHours.pcap_ISCX.csv
  [unseen] Thu-AM-WebAttacks    Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  [unseen] Thu-PM-Infiltration  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  [unseen] Fri-AM-Bot           Friday-WorkingHours-Morning.pcap_ISCX.csv
  [unseen] Fri-PM-PortScan      Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  [unseen] Fri-PM-DDoS          Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


[]

In [7]:
# Cell 3: build the splits (writes data/processed_m2/)
train, val, test_seen, test_unseen = run_m2_split()

Looking in /Users/skishore/Documents/GitHub/CSCI6330-Detector-Project/data/raw
  [seen  ] Monday               Monday-WorkingHours.pcap_ISCX.csv
  [seen  ] Tuesday              Tuesday-WorkingHours.pcap_ISCX.csv
  [seen  ] Wednesday            Wednesday-workingHours.pcap_ISCX.csv
  [unseen] Thu-AM-WebAttacks    Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  [unseen] Thu-PM-Infiltration  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  [unseen] Fri-AM-Bot           Friday-WorkingHours-Morning.pcap_ISCX.csv
  [unseen] Fri-PM-PortScan      Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  [unseen] Fri-PM-DDoS          Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Loading and de-duplicating seen days...
 Monday
  dropped 26935 duplicate rows (529918 -> 502983)
 Tuesday
  dropped 24065 duplicate rows (445909 -> 421844)
 Wednesday
  dropped 81909 duplicate rows (692703 -> 610794)

Loading and de-duplicating unseen days...
 Thu-AM-WebAttacks
  dropped 6066 dupl

In [8]:
# Cell 4: multi-seed validation report (train/val only, nothing saved)
from src import m2_experiments as m2
allres, summ = m2.run_val()


########## seed 42 ##########
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286

=== validation, seed 42 ===

AUROC
detector       n  Autoencoder  RF-IDS
item                                 
overall   230222       0.9888  0.9998

RECALL by attack family
                      n  Autoencoder @0.1%  Autoencoder @1.0%  RF-IDS @0.1%  RF-IDS @1.0%
item                                                                                     
ALL attacks       30432             0.8391             0.8946        0.9940        0.9995
DoS GoldenEye      1542             0.2575             0.4754        0.9604        0.9948
DoS Hulk          25927             0.9544             0.9735        0.9991        0.9997
DoS Slowhttptest    784             0.0612             0.9120        0.8763        0.9987
Do

In [11]:
from src import diagnostics as dg
_ = dg.port_only_check("data/processed_m2")   # is FTP/SSH solvable by port alone?
_ = dg.error_analysis("data/processed_m2")    # why does the AE miss brute force?
_ = dg.rf_leakage_check("data/processed_m2")  # RF still ~1.0 with top features removed?

=== Destination Port ONLY (RF, 50 trees) - AUROC 0.9842 ===
  distinct score values on val: 5
  target FPR  achieved FPR   recall  benign tied @thr
          1%        0.0011   0.0292            0.0299
          5%        0.0311   1.0000            0.9689
         10%        0.0311   1.0000            0.9689
         20%        0.0311   1.0000            0.9689

  Top destination ports by share of each class (val):
    benign: 53=61.2%, 443=6.2%, 80=3.0%, 88=0.5%, 389=0.4%
    attack: 80=95.5%, 21=2.9%, 22=1.6%, 444=0.0%
  If benign port 80 > 1% and attack port 80 is high, the 1% threshold sits
  on port 80's score, strict `>` flags nothing, and 0% recall is a tie artifact.
--- Autoencoder ---
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286

AE threshold 0.0449, achieved FPR 0.0100
att

In [1]:
from src import diagnostics as dg
abl = dg.context_ablation("data/processed_m2", seeds=(42,))


--- RF-IDS | without port (current) | seed 42 ---

--- Autoencoder | without port (current) | seed 42 ---
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286

--- RF-IDS | with port | seed 42 ---

--- Autoencoder | with port | seed 42 ---
  epoch   1/30  train MSE: 0.186104
  epoch   5/30  train MSE: 0.053016
  epoch  10/30  train MSE: 0.041032
  epoch  15/30  train MSE: 0.031367
  epoch  20/30  train MSE: 0.023586
  epoch  25/30  train MSE: 0.017405
  epoch  30/30  train MSE: 0.022291

=== Context ablation (val) ===
      model  seed                variant  auroc    thr  achieved_fpr  recall@1%
     RF-IDS    42 without port (current) 0.9998 0.0150        0.0082     0.9995
Autoencoder    42 without port (current) 0.9888 0.0449        0.0100     0.8946
     RF-IDS    42              with 

In [2]:
from src import m2_experiments as m2
dets2, thr2 = m2.load_official()
val = m2._load("val")
y = val["y"]
for name, d in dets2.items():
    flag = d["score"](val["X"]) > thr2[name][0.01]
    tp = int((flag & (y == 1)).sum()); fp = int((flag & (y == 0)).sum())
    fn = int((~flag & (y == 1)).sum()); tn = int((~flag & (y == 0)).sum())
    n = tp + fp + fn + tn
    print(name, "acc %.3f prec %.3f rec %.4f f1 %.3f fpr %.4f fnr %.4f" % (
        (tp+tn)/n, tp/(tp+fp), tp/(tp+fn), 2*tp/(2*tp+fp+fn), fp/(fp+tn), fn/(fn+tp)))

Autoencoder acc 0.977 prec 0.932 rec 0.8946 f1 0.913 fpr 0.0100 fnr 0.1054
RF-IDS acc 0.993 prec 0.949 rec 0.9995 f1 0.973 fpr 0.0082 fnr 0.0005


In [3]:
import time, numpy as np
from pathlib import Path
from src import m2_experiments as m2
from src.autoencoder import fit_scaler, train_autoencoder
from src.rf_ids import train_rf

train, val = m2._load("train"), m2._load("val")
Xb = train["X"][train["y"] == 0]

# training time, 3 runs each (seed 42)
ae_t, rf_t = [], []
for _ in range(3):
    t = time.perf_counter(); sc = fit_scaler(Xb); train_autoencoder(sc.transform(Xb), seed=42); ae_t.append(time.perf_counter() - t)
    t = time.perf_counter(); train_rf(train["X"], train["y"], seed=42); rf_t.append(time.perf_counter() - t)
print("training s  AE %.1f +/- %.1f | RF %.1f +/- %.1f" % (np.mean(ae_t), np.std(ae_t), np.mean(rf_t), np.std(rf_t)))

# inference per 1,000 flows on the official models: 1 warm-up, then 5 timed passes over validation
dets, _ = m2.load_official()
X = val["X"]; n = len(X)
for name, d in dets.items():
    d["score"](X[:5000])
    ts = []
    for _ in range(5):
        t = time.perf_counter(); d["score"](X); ts.append(time.perf_counter() - t)
    per1k = np.array(ts) / (n / 1000) * 1000   # ms per 1,000 flows
    print("%s inference %.3f +/- %.3f ms per 1,000 flows (n=%d)" % (name, per1k.mean(), per1k.std(), n))

# saved size
for f in ["autoencoder.pt", "scaler.joblib", "rf.joblib"]:
    print(f, "%.1f KB" % (Path("models_m2", f).stat().st_size / 1024))

  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286
  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286
training s  AE 106.0 +/- 4.9 | RF 76.9 +/- 0.4
Autoencoder inference 0.373 +/- 0.164 ms per 1,000 flows (n=230222)
RF-IDS inference 1.116 +/- 0.008 ms per 1,000 flows (n=230222)
autoencoder.pt 66.6 KB
scaler.joblib 2.4 KB
rf.joblib 43364.6 KB


In [12]:
# Cell 5: ALREADY RUN, DO NOT RERUN. Official models + thresholds
# dets, thr = m2.save_official()

  epoch   1/30  train MSE: 0.176067
  epoch   5/30  train MSE: 0.057784
  epoch  10/30  train MSE: 0.039552
  epoch  15/30  train MSE: 0.034574
  epoch  20/30  train MSE: 0.024864
  epoch  25/30  train MSE: 0.019505
  epoch  30/30  train MSE: 0.016286

=== validation, official seed 42 ===

AUROC
detector       n  Autoencoder  RF-IDS
item                                 
overall   230222       0.9888  0.9998

RECALL by attack family
                      n  Autoencoder @0.1%  Autoencoder @1.0%  RF-IDS @0.1%  RF-IDS @1.0%
item                                                                                     
ALL attacks       30432             0.8391             0.8946        0.9940        0.9995
DoS GoldenEye      1542             0.2575             0.4754        0.9604        0.9948
DoS Hulk          25927             0.9544             0.9735        0.9991        0.9997
DoS Slowhttptest    784             0.0612             0.9120        0.8763        0.9987
DoS slowloris       807 

In [13]:
print(thr)

{'Autoencoder': {0.01: 0.04489630088210106, 0.001: 0.26539134979248047}, 'RF-IDS': {0.01: 0.015, 0.001: 0.175}}


In [14]:
import importlib
from src import m2_experiments as m2
importlib.reload(m2)
print("new version loaded:", hasattr(m2, "reanalyze_saved"))

new version loaded: True


In [15]:
dets2, thr2 = m2.load_official()
print(thr2)   # should equal the thresholds you pasted
val = m2._load("val")
m2.show(m2.evaluate_split(dets2, thr2, val), "validation via load_official (round-trip check)")

{'Autoencoder': {0.01: 0.04489630088210106, 0.001: 0.26539134979248047}, 'RF-IDS': {0.01: 0.015, 0.001: 0.175}}

=== validation via load_official (round-trip check) ===

AUROC, all flows
detector       n  Autoencoder  RF-IDS
item                                 
overall   230222       0.9888  0.9998

AUROC by day (that day's attacks vs that day's benign)
detector       n  Autoencoder  RF-IDS
item                                 
Tuesday    63249       0.8946  0.9999
Wednesday  91589       0.9934  0.9997

AUROC by attack family (family vs benign flows from the same day)
detector              n  Autoencoder  RF-IDS
item                                        
DoS GoldenEye      1542       0.9696  0.9979
DoS Hulk          25927       0.9954  0.9998
DoS Slowhttptest    784       0.9916  0.9994
DoS slowloris       807       0.9768  0.9999
FTP-Patator         889       0.9016  0.9999
Heartbleed            1       0.9997  0.9994
SSH-Patator         482       0.8815  0.9999

RECALL by attack f

In [10]:
# Cell 6: DO NOT RUN YET. Scores the test sets, once
# m2.score_test()